*  DSC550-T302 Data Mining (2265-1)
*  Term Project: Term Project Milestone 3: Model Building and Evaluation
*  David Jorgensen

# Term Project

## Project Description

Water scarcity is one of the defining resource challenges facing the American West. Utah, as an arid state that depends heavily on snowpack-fed rivers and reservoirs, is particularly vulnerable. At the same time, Utah has emerged as a rapidly growing technology and energy hub, attracting large-scale data centers drawn by land availability, tax incentives, and access to power. Nuclear energy is also being revisited as a low-carbon baseload option, with proposed sites in the region. Both data centers and nuclear power plants are significant industrial water consumers: nuclear plants use water for steam generation and cooling, while data centers rely on evaporative cooling systems that can consume millions of gallons per day at scale. As these facilities grow in number and capacity, their cumulative draw on Utah's already-stressed watersheds becomes a pressing concern for water managers, state regulators, and communities that depend on those same water sources.

This project addresses the following question: **to what extent does thermoelectric water consumption, from nuclear power plants and large industrial facilities such as data centers, reduce effective water availability in Utah watersheds, and can we predict the impact of future growth in that consumption?** Using watershed-level water availability and consumption data across Utah's 2,558 HUC12 watersheds from 2010 to 2019, alongside thermoelectric water withdrawal and consumption records for the same watersheds and time period, the goal is to quantify the relationship between industrial thermoelectric demand and watershed water availability.

The **target variable** for the predictive model is net water availability at the watershed level, defined as the difference between natural water availability (mm/month) and total water consumption (mm/month). This metric captures how much usable water remains in a watershed after human demand, including thermoelectric consumption, is accounted for. By building a regression model with thermoelectric consumption (in million gallons per day) as a key predictor, alongside year and watershed-level controls for natural variability, the model will estimate how each additional unit of thermoelectric demand reduces net availability. The resulting model can then be applied to project water stress scenarios as data center capacity and nuclear output expand in Utah over the coming decades.

The primary stakeholders for this analysis are Utah water resource managers, the Utah Division of Water Resources, environmental planning agencies, and technology companies conducting water impact assessments for proposed facilities. Accurate predictions of thermoelectric water stress will support sustainable siting decisions and long-term water allocation planning in a state where demand is rising and supply is increasingly uncertain.

**Data Sources**

- United States Geological Survey (USGS):
  - [Model: Thermoelectric Power Water-Use Model](https://water.usgs.gov/nwaa-data/data-catalog/sector/water-use/model/wu-thermoelectric/)
  - [Model: National Water Availability Assessment Outputs (CONUS 2025)](https://water.usgs.gov/nwaa-data/data-catalog/sector/integrated-water-availability/model/iwa-assessment-outputs-conus-2025/)
- Western Resource Advocates (WRA):
  - [Data Center Impacts in the West: Policy Solutions for Water and Energy Use](https://westernresourceadvocates.org/wp-content/uploads/2025/08/2025_DataCenter_FactSheet_UT.pdf)
- Salt Lake Tribune (SLT):
  - [Can Utah become a data center hub without draining its water supply?](https://www.sltrib.com/news/politics/2026/01/12/does-utah-have-enough-water-be/)
- Stanford University:
  - [Nuclear Reactor Water Usage and the Implications of Limited Water Availability](http://large.stanford.edu/courses/2013/ph241/abdul-khabir2/)
- University of Utah (UofU):
  - [Utah’s 2025 Energy Landscape and Outlook for 2026 and Beyond](https://d36oiwf74r1rap.cloudfront.net/wp-content/uploads/2026/04/ERG2026-Energy-RB-Apr2026.pdf)
- PacifiCorp:
  - [2025 Integrated Resource Plan: Volume 1](https://www.pacificorp.com/content/dam/pcorp/documents/en/pacificorp/energy/integrated-resource-plan/2025-irp/2025_IRP_Vol_1.pdf)
- Utah Clean Energy Organization (UCE):
  - [Estimated Emissions and Water Consumption from the Proposed Stratos Data Center](https://utahcleanenergy.org/estimated-emissions-and-water-consumption-from-the-proposed-stratos-data-center/)
- Deseret News
  - [Data centers on the way to Utah, but how fast and at what cost? ](https://www.deseret.com/business/2026/04/15/utah-data-centers-artificial-intelligence-ai-water-energy-electricity-use-economic-impacts-jobs-tax-incentives/)
- ABC4 News
  - [Tracking where AI data centers are being built in Utah: Interactive map](https://www.abc4.com/news/digital-exclusives/tracking-ai-data-centers-utah-map/)

## Previous Work

In [59]:
import re, json, csv, os
import numpy as np
import pandas as pd
import requests
from sklearn.preprocessing import StandardScaler
import geopandas as gpd
from pygeohydro import WBD
from shapely.geometry import Point, box as shapely_box

In [60]:
utah_water = pd.concat([
    pd.read_csv(f'data/utah-water/nwaa_data_{i}_of_5.csv') for i in range(1, 6)
], ignore_index=True)

thermo_water = pd.concat([
    pd.read_csv(f'data/thermoelectric-water/nwaa_data_{i}_of_5(1).csv') for i in range(1, 6)
], ignore_index=True)

In [61]:
water = pd.merge(utah_water, thermo_water, on=['huc12_id', 'year'], how='left')
water[['tewdftot_mgd', 'tecuftot_mgd']] = water[['tewdftot_mgd', 'tecuftot_mgd']].fillna(0)

In [62]:
pacificorp = pd.read_csv('data/pacificorp-power-water-consumption/table_7_4_thermal_supply_side_2025.csv')
pacificorp['water_gal_mwh'] = pd.to_numeric(pacificorp['Water Consumed (Gal/MWh)'], errors='coerce')

In [63]:
uofu = pd.read_csv('data/uofu_energy_data/UofU_energy_data.csv')

In [64]:
dc = pd.read_csv('data/utah-data-centers/utah_datacenters.csv')

DC_OPERATIONAL_MW    = 920
DC_CONSTRUCTION_MW   = 2600
DC_STRATOS_MW        = 9000
DC_TOTAL_PIPELINE_MW = DC_OPERATIONAL_MW + DC_CONSTRUCTION_MW + DC_STRATOS_MW

### Data Center Inventory: Context and Water Demand Estimation

Utah's data center pipeline is remarkable in scale. The 9,000 MW Stratos project alone is nearly **10× the state's entire current operational data center capacity** (920 MW). Key observations from the inventory:

- **Geography:** Current operational facilities concentrate in Salt Lake Valley (West Jordan, Bluffdale, West Valley City). Future large-scale projects spread to rural counties: Millard (Delta), Duchesne, Box Elder, Iron — counties with smaller existing water infrastructure.
- **Power sourcing:** Large proposed facilities are planning off-grid, on-site generation (gas, solar, nuclear) because the Utah grid cannot support their load. This is materially significant: on-site gas generation adds thermoelectric water demand on top of cooling water demand.
- **Stratos controversy:** The original permit application described 9 GW of on-site gas generation; O'Leary stated at the May 2026 public hearing that it would instead use "solar, wind, and batteries." If gas generation is retained, Stratos's on-site gas turbines alone would consume an estimated **~50–70 MGD** of cooling water (SCCT/CCCT range from PacifiCorp table).
- **Joule Power Delta:** Claims "90% less water than traditional" due to advanced recovery systems; plans to scale to 12 GW via nuclear by 2032. Nuclear at 720 Gal/MWh × 12,000 MW → ~280 MGD, even with recovery systems.
- **Water reporting:** Only 8 of 17 catalogued facilities have measured water consumption data (WRA 2025). The remaining 9 — including the largest proposed projects — have no public water use figures.

**Estimation approach:** Water Usage Effectiveness (WUE, L/kWh) is the standard metric for data center water intensity. A WUE of 1.5 L/kWh (~0.396 Gal/kWh) is typical for evaporative cooling in an arid western climate. A range of 0.5–2.0 L/kWh brackets low (air-cooled) to high (evaporative) scenarios.

In [65]:
CF        = 0.80
L_PER_GAL = 3.785

def dc_water_mgd(mw, wue_l_per_kwh, cf=CF):
    """MW capacity → MGD cooling water consumption at given WUE (L/kWh)."""
    kwh_per_day = mw * 1000 * cf * 24
    return kwh_per_day * wue_l_per_kwh / L_PER_GAL / 1_000_000

scenarios = {
    'low':  0.5,   # WUE=0.5 L/kWh, air-cooled
    'mid':  1.5,   # WUE=1.5 L/kWh, evaporative (arid-climate typical)
    'high': 2.0,   # WUE=2.0 L/kWh, hot/arid worst-case
}

### Data Center Water Demand: Key Findings

The WUE-based estimates reveal the order-of-magnitude significance of the pipeline:

- **Current 920 MW** already consumes an estimated **2–9 MGD** for cooling alone — consistent with the WRA-measured facilities (0.94 MGD for 8 named Salt Lake Valley sites) representing a subset of the total fleet.
- **Under-construction 2,600 MW** adds another **6–25 MGD**, roughly matching or exceeding the statewide USGS thermoelectric baseline of ~36–43 MGD.
- **Stratos 9,000 MW** alone — if fully built and using evaporative cooling — would consume **21–87 MGD**, comparable to the entire historical thermoelectric baseline in a single facility.
- **Total 12,520 MW pipeline** at mid-estimate (~50 MGD cooling) would nearly **double** current statewide thermoelectric water consumption.

These estimates cover **cooling water only**. Facilities with on-site gas generation (Stratos, Summit Ridge, Antelope, Nine Mile, Joule Delta Phase 1) add significant additional thermoelectric water for power generation cooling — potentially 50–70 MGD for Stratos alone at 9 GW of SCCT/CCCT gas generation.

**Role in modeling:** The data center inventory is not directly used as a feature in the 2010–2019 regression model (historical data centers are captured implicitly in `industrial_gwh`). Instead, it informs the **projection scenarios** for Milestone 3: the model's estimated relationship between thermoelectric consumption and net water availability will be applied at projected future consumption levels derived from the data center pipeline above.

### Additional Data Source: Cleanview Geocoded Inventory

The manually assembled inventory captures 17 named facilities from WRA and ABC4 reporting, but lacks precise geocoordinates needed for watershed-level assignment. [Cleanview](https://www.cleanview.co/data-centers/utah) tracks 29 Utah data centers (13 operating, 16 planned) with facility-level lat/lon and capacity (MW).

The site is a Next.js app that delivers facility data as React Server Component (RSC) payloads embedded in inline `<script>` tags. The markers array — containing id, name, latitude, longitude, status, capacityMw, and detail for every facility — is JSON-encoded inside a `self.__next_f.push([1, "..."])` call. Fetching the page with `Accept: text/x-component` triggers the RSC response, which contains this payload. Parsing it with `json.loads()` after identifying the containing chunk gives the full dataset without requiring a headless browser.

HUC12 watershed boundaries are downloaded from USGS via the `pygeohydro` WBD client and spatially joined to facility points using `geopandas`. This produces a direct mapping from each data center to the USGS HUC12 code used in the NWAA watershed model, enabling watershed-specific projection scenarios in Milestone 3.

In [ ]:
CLEANVIEW_CSV = 'data/utah-data-centers/cleanview_utah_datacenters.csv'

if os.path.exists(CLEANVIEW_CSV):
    dc_cleanview = pd.read_csv(CLEANVIEW_CSV)
else:
    resp = requests.get(
        'https://www.cleanview.co/data-centers/utah',
        headers={'Accept': 'text/x-component'},
        timeout=30
    )
    resp.raise_for_status()
    chunks = re.findall(r'self\.__next_f\.push\(\[1,(.*?)\]\)</script>', resp.text, re.DOTALL)
    markers = None
    for chunk in chunks:
        if 'latitude' not in chunk:
            continue
        decoded = json.loads(chunk)
        idx = decoded.find('"markers":[')
        if idx < 0:
            continue
        start = idx + len('"markers":')
        depth, end = 0, start
        for j, c in enumerate(decoded[start:], start):
            if c == '[':   depth += 1
            elif c == ']':
                depth -= 1
                if depth == 0:
                    end = j + 1
                    break
        markers = json.loads(decoded[start:end])
        break
    if markers is None:
        raise RuntimeError("Cleanview RSC payload structure changed — re-inspect network response")
    os.makedirs('data/utah-data-centers', exist_ok=True)
    with open(CLEANVIEW_CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['id','name','status','capacity_mw','latitude','longitude','detail'])
        w.writeheader()
        for m in markers:
            w.writerow({'id': m['id'], 'name': m['name'], 'status': m['status'],
                        'capacity_mw': m.get('capacityMw',''), 'latitude': m['latitude'],
                        'longitude': m['longitude'], 'detail': m.get('detail','')})
    dc_cleanview = pd.read_csv(CLEANVIEW_CSV)

,id,name,status,capacity_mw,latitude,longitude,detail
0,1034,Aligned Salt Lake SLC 01,Operating,34.0,40.586273,-111.970101,"Salt Lake, UT"
1,986,Aligned Salt Lake SLC 02,Operating,48.0,40.583715,-111.968525,"Salt Lake, UT"
2,911,Aligned Salt Lake SLC 03,Operating,100.0,40.581946,-111.968812,"Salt Lake, UT"
3,1350,Aligned Salt Lake SLC 04,Operating,NaN,40.581946,-111.968812,"Salt Lake, UT"
4,1293,Downtown Salt Lake City Data Center: SLC1,Operating,1.0,40.768522,-111.885873,"Salt Lake, UT"
5,1349,Eagle Mountain Data Center,Operating,NaN,40.264863,-112.015144,"Utah, UT"
6,1292,EdgeConneX SLC01,Operating,1.0,40.720857,-111.985032,"Salt Lake, UT"
7,1255,Granite Point Data Center: SLC2,Operating,3.0,40.479629,-111.904597,"Salt Lake, UT"
8,1191,Granite Point Data Center: SLC3,Operating,7.0,40.479833,-111.905764,"Salt Lake, UT"
9,1236,Granite Point Data Center: SLC4,Operating,4.0,40.481391,-111.905009,"Salt Lake, UT"


In [ ]:
HUC12_GPKG = 'data/huc12/utah_huc12.gpkg'
HUC12_CSV  = 'data/utah-data-centers/cleanview_utah_datacenters_huc12.csv'

if os.path.exists(HUC12_GPKG):
    huc12 = gpd.read_file(HUC12_GPKG)
else:
    os.makedirs('data/huc12', exist_ok=True)
    utah_bbox = shapely_box(-114.5, 36.9, -108.9, 42.1)
    huc12 = WBD("huc12").bygeom(utah_bbox, geo_crs=4326)
    huc12.to_file(HUC12_GPKG, driver='GPKG')

if huc12.crs.to_epsg() != 4326:
    huc12 = huc12.to_crs(4326)

dc_gdf = gpd.GeoDataFrame(
    dc_cleanview,
    geometry=[Point(lon, lat) for lon, lat in zip(dc_cleanview['longitude'], dc_cleanview['latitude'])],
    crs=4326
)

dc_huc12 = (
    gpd.sjoin(dc_gdf, huc12[['huc12','name','areasqkm','geometry']], how='left', predicate='within')
    .drop_duplicates(subset='id')
    [['id','name_left','status','capacity_mw','latitude','longitude','detail','huc12','name_right','areasqkm']]
    .rename(columns={'name_left': 'facility', 'name_right': 'huc12_name'})
    .reset_index(drop=True)
)
dc_huc12.to_csv(HUC12_CSV, index=False)

,id,facility,status,capacity_mw,latitude,longitude,detail,huc12,huc12_name,areasqkm
0,1034,Aligned Salt Lake SLC 01,Operating,34.0,40.586273,-111.970101,"Salt Lake, UT",160202040106,Bingham Creek,80.05
1,986,Aligned Salt Lake SLC 02,Operating,48.0,40.583715,-111.968525,"Salt Lake, UT",160202040106,Bingham Creek,80.05
2,911,Aligned Salt Lake SLC 03,Operating,100.0,40.581946,-111.968812,"Salt Lake, UT",160202040106,Bingham Creek,80.05
3,1350,Aligned Salt Lake SLC 04,Operating,NaN,40.581946,-111.968812,"Salt Lake, UT",160202040106,Bingham Creek,80.05
4,1293,Downtown Salt Lake City Data Center: SLC1,Operating,1.0,40.768522,-111.885873,"Salt Lake, UT",160202040306,Red Butte Creek-Emigration Creek,72.00
5,1349,Eagle Mountain Data Center,Operating,NaN,40.264863,-112.015144,"Utah, UT",160202010405,Mercer Canyon,119.73
6,1292,EdgeConneX SLC01,Operating,1.0,40.720857,-111.985032,"Salt Lake, UT",160202040307,Parleys Creek-Jordan River,124.95
7,1255,Granite Point Data Center: SLC2,Operating,3.0,40.479629,-111.904597,"Salt Lake, UT",160202040105,Corner Canyon-Jordan River,54.17
8,1191,Granite Point Data Center: SLC3,Operating,7.0,40.479833,-111.905764,"Salt Lake, UT",160202040105,Corner Canyon-Jordan River,54.17
9,1236,Granite Point Data Center: SLC4,Operating,4.0,40.481391,-111.905009,"Salt Lake, UT",160202040105,Corner Canyon-Jordan River,54.17


### Cleanview + HUC12 Geocoding: Key Findings

The spatial join maps all 23 Cleanview facilities to USGS HUC12 watershed codes, enabling direct linkage to the NWAA watershed model used throughout this project. Four watersheds absorb nearly all planned capacity:

| HUC12 | Watershed | Planned MW | Notes |
|---|---|---|---|
| `160300050805` | Sugarville | 10,050 | Creekstone Delta Gigasite + BluSky; Millard County, Great Basin |
| `160203090501` | Headwaters Hansel Valley Wash | 9,000 | Stratos (Wonder Valley); Box Elder County, Great Basin |
| `160300051502` | Whiskey Creek | 4,000 | HPC Campus; Millard County, Great Basin |
| `160202040106` | Bingham Creek | 147 | West Jordan phases 2–3; Salt Lake County, Jordan River basin |

All three high-capacity rural sites (Sugarville, Hansel Valley, Whiskey Creek) are in **endorheic Great Basin watersheds** — closed basins with no outlet to the ocean. Water withdrawn from these systems does not return downstream; consumptive loss is total. This makes the water stress implications especially acute relative to equivalent consumption in a connected river basin.

The `huc12` column in `dc_huc12` now provides the foreign key to join proposed facility demand directly to the USGS NWAA watershed records (`water['huc12_id']`) for Milestone 3 projection scenarios.

### Clean and Derive Features from UofU Energy Data

The UofU dataset covers 2000–2025 with statewide Utah electricity generation by fuel type (GWh) and consumption by sector. The 2025 row is labeled `2025p` (preliminary), which needs to be stripped to a numeric year. Several derived features are useful for modeling:

- **`renewable_fraction`**: share of generation from non-combustion sources (hydro, wind, solar, biomass, geothermal). Rising renewable share means reduced thermoelectric water demand from new capacity.
- **`fossil_fraction`**: share from coal + petroleum + natural gas. Inversely related to renewable fraction but useful separately since coal and gas have very different water intensities.
- **`industrial_gwh`**: industrial sector electricity consumption. Serves as a proxy for data center load — data centers are classified as industrial consumers and have been the primary driver of Utah industrial electricity growth since ~2016.
- **`industrial_yoy_growth`**: year-over-year growth rate of industrial consumption. Captures acceleration of data center buildout.

Only years 2010–2019 overlap with the USGS watershed data and will be used for model training. The full 2000–2025 series is retained for projection scenarios.

In [68]:
uofu['Year'] = uofu['Year'].astype(str).str.replace('p', '', regex=False).astype(int)

renewable_cols = ['Hydro', 'Wind', 'Solar', 'Biomass', 'Geothermal']
fossil_cols    = ['Coal', 'Petroleum', 'Natural Gas']

uofu['renewable_fraction']    = uofu[renewable_cols].sum(axis=1) / uofu['Net Generation Total (GWh)']
uofu['fossil_fraction']       = uofu[fossil_cols].sum(axis=1)    / uofu['Net Generation Total (GWh)']
uofu['industrial_gwh']        = uofu['Industrial Consumption (GWh)']
uofu['industrial_yoy_growth'] = uofu['industrial_gwh'].pct_change()

### Build Water Intensity Lookup from PacifiCorp Table

The PacifiCorp table provides water consumed per MWh generated (Gal/MWh) for each planned technology. This is the standard metric for comparing thermoelectric water demand across fuel types. Key values from the table:

| Technology | Gal/MWh | Notes |
|---|---|---|
| Natural Gas SCCT (simple cycle) | 27–28 | Air-cooled, low water need |
| Natural Gas CCCT (combined cycle, dry) | 210 | Recirculating cooling |
| Natural Gas CCCT + 95% CCS | 323 | CCS adds ~54% more water |
| Geothermal NF-EGS Binary | 510 | Flash/binary cycle |
| Nuclear SMR | 720 | High cooling demand |
| Coal + 95% CCS | 186 | Post-retrofit basis |

**Important caveat:** These are design-specification values for *new* planned builds. Existing Utah plants (mostly pre-CCS coal and gas) operate under different configurations. The USGS thermoelectric dataset already captures *actual* consumption for 2010–2019 at the watershed level. The PacifiCorp intensity values are primarily used here to estimate future water demand under energy-mix scenarios (e.g., adding nuclear SMR capacity) and to interpret the relative water cost of different technologies.

For mapping to the UofU historical generation data, representative intensity values are assigned per fuel category:
- **Coal (existing, wet recirculating):** ~530 Gal/MWh (industry standard for once-through wet towers; not in PacifiCorp table which shows CCS retrofits only)
- **Natural Gas (mixed fleet):** ~150 Gal/MWh (blend of SCCT and CCCT plants)
- **Geothermal:** 510 Gal/MWh (from PacifiCorp NF-EGS binary)
- **Nuclear:** 720 Gal/MWh (PacifiCorp SMR; currently zero capacity in Utah)

In [69]:
water_intensity_lookup = {
    'Coal':        530,   # existing Utah fleet, wet recirculating towers
    'Natural Gas': 150,   # blended SCCT/CCCT fleet average
    'Geothermal':  510,   # PacifiCorp NF-EGS Binary (IRP 2025 Table 7.4)
    'Nuclear':     720,   # PacifiCorp SMR (IRP 2025 Table 7.4)
    'Petroleum':   200,
    'Biomass':     27,
    'Hydro':       0,
    'Wind':        0,
    'Solar':       0,
}

### Estimate Statewide Thermoelectric Water Consumption and Validate Against USGS

Applying water intensity factors to the UofU generation data produces an independent estimate of statewide thermoelectric water consumption for each year. This estimate is then compared against the USGS-reported totals from the watershed model (summed across all Utah HUC12 watersheds).

**Why this matters:** If the two estimates are in reasonable agreement during the 2010–2019 overlap period, it validates the intensity-factor approach for extrapolating beyond 2019 (where USGS data ends) using UofU generation projections. Systematic disagreement would indicate that the existing Utah fleet operates at different efficiencies than the lookup values, requiring calibration.

**Unit conversion:** UofU data is in GWh; intensity factors are in Gal/MWh.  
`GWh × 1,000 MWh/GWh × Gal/MWh ÷ 1,000,000 gal/MGal ÷ 365 days/year = MGD`

In [ ]:
fuel_to_col = {
    'Coal': 'Coal', 'Natural Gas': 'Natural Gas', 'Geothermal': 'Geothermal',
    'Nuclear': None, 'Petroleum': 'Petroleum', 'Biomass': 'Biomass',
}
uofu_est = uofu.copy()
uofu_est['est_thermo_consumption_mgd'] = 0.0
for fuel, col in fuel_to_col.items():
    if col is None:
        continue
    uofu_est['est_thermo_consumption_mgd'] += uofu_est[col] * 1000 * water_intensity_lookup[fuel] / 1e6 / 365

usgs_yearly = water.groupby('year')['tecuftot_mgd'].sum().reset_index()
usgs_yearly.columns = ['Year', 'usgs_actual_mgd']

comparison = pd.merge(uofu_est[['Year','est_thermo_consumption_mgd']], usgs_yearly, on='Year', how='left')
comparison['ratio'] = comparison['est_thermo_consumption_mgd'] / comparison['usgs_actual_mgd']

,Year,est_thermo_consumption_mgd,usgs_actual_mgd,ratio
10,2010,52.523951,42.803077,1.227107
11,2011,50.773167,40.803074,1.244347
12,2012,47.920384,41.677980,1.149777
13,2013,52.963718,43.931231,1.205605
14,2014,52.655345,43.109867,1.221422
15,2015,49.961575,42.481160,1.176088
16,2016,41.937912,36.145329,1.160258
17,2017,41.431140,35.926544,1.153218
18,2018,41.860145,37.791195,1.107669
19,2019,40.961910,35.843539,1.142798


### Interpretation

The estimated totals (from UofU generation × intensity factors) are likely to be significantly higher than the USGS actuals. This is expected for two reasons:

1. **Coverage scope:** The USGS thermoelectric model reports only consumption *attributable to specific watershed grid cells* in Utah. It does not cover all thermoelectric plants statewide — particularly plants whose water withdrawals are diffuse or whose HUC12 attribution is ambiguous.
2. **Technology mix:** The intensity factors are point estimates for representative configurations. Utah's existing coal fleet uses wet recirculating towers but operates at varying capacity factors; coal plants that ran at reduced loads (especially post-2016 as coal declined) would show lower actual consumption than the fixed-rate estimate implies.

**Decision:** The estimated values confirm the relative order of magnitude and validate the fuel-type hierarchy (coal is dominant, gas growing, geothermal small). The ratio can be used as a scaling calibration factor for projections beyond 2019. The USGS actuals remain the authoritative inputs for the 2010–2019 model training period.

### Merge Statewide Energy Features into Watershed Dataset

The watershed-level `water` DataFrame (25,580 rows × year 2010–2019) is enriched with statewide energy features from UofU. Since these features are constant across all watersheds within a given year, they are joined on `year`. This captures the macro-level energy mix context that shapes statewide thermoelectric water pressure — information that is not encoded in per-watershed USGS fields alone.

Features added from UofU:
- `renewable_fraction`: measures how much of Utah's grid is non-thermoelectric (lower thermoelectric pressure years)
- `fossil_fraction`: counterpart, drives bulk of thermoelectric water demand
- `industrial_gwh`: proxy for data center and heavy industrial load growth
- `industrial_yoy_growth`: rate of industrial demand acceleration

In [71]:
uofu_join = uofu[['Year','renewable_fraction','fossil_fraction',
                   'industrial_gwh','industrial_yoy_growth']].rename(columns={'Year': 'year'})
water = pd.merge(water, uofu_join, on='year', how='left')

### Feature Engineering

**Target variable:** `net_avail_mm_mo = availab_mm/mo − consum_mm/mo`

This is the core outcome of interest: how much usable water remains in a watershed after all human consumption is subtracted from natural availability. Negative values indicate water stress (demand exceeds supply). The model will predict this quantity given thermoelectric consumption and contextual covariates.

**Features:**

| Feature | Type | Description |
|---|---|---|
| `tecuftot_mgd` | Continuous | Thermoelectric water consumption (key predictor) |
| `tewdftot_mgd` | Continuous | Thermoelectric water withdrawal (correlated but distinct — some withdrawn water returns) |
| `availab_mm_mo` | Continuous | Natural water availability (precipitation + snowmelt signal; major confounder) |
| `fossil_fraction` | Continuous | Share of generation from fossil fuels (statewide thermoelectric pressure) |
| `industrial_gwh` | Continuous | Industrial electricity consumption (data center growth proxy) |
| `year` | Ordinal | Captures secular trend in infrastructure buildout and climate |
| `has_thermo` | Binary | Whether the watershed contains a thermoelectric facility |

**Not included:**
- `consum_mm/mo`: directly used in target construction, would cause data leakage
- `huc12_id`: 2,558 unique watershed IDs — too many dummies; watershed fixed effects will be handled via model architecture if needed
- `industrial_yoy_growth`: collinear with `industrial_gwh`; also NaN for 2010 (first row)
- `renewable_fraction`: collinear with `fossil_fraction` (they sum to ~1); dropping to avoid multicollinearity

In [72]:
water['net_avail_mm_mo'] = water['availab_mm/mo'] - water['consum_mm/mo']
water = water.rename(columns={'availab_mm/mo': 'availab_mm_mo'})
water['has_thermo'] = (water['tecuftot_mgd'] > 0).astype(int)

feature_cols = [
    'tecuftot_mgd', 'tewdftot_mgd', 'availab_mm_mo',
    'fossil_fraction', 'industrial_gwh', 'year', 'has_thermo',
]
target_col = 'net_avail_mm_mo'
model_df = water[feature_cols + [target_col, 'huc12_id']].dropna()

### Temporal Train/Test Split

**Why temporal, not random:** This dataset is a panel of watershed × year observations. Random splitting would place 2019 data in training and 2017 data in test — the model would then predict "past" conditions from "future" information, producing optimistic accuracy that doesn't reflect real deployment. Since the goal is to project water stress under future growth scenarios, the split must respect the arrow of time.

**Split boundary:** 2010–2017 → training (8 years × 2,558 watersheds = 20,464 rows)  
2018–2019 → test (2 years × 2,558 watersheds = 5,116 rows)

This gives a ~80/20 split temporally. The two test years include 2018 (the driest year in the dataset, high stress) and 2019 (a wet recovery year), which tests the model across contrasting hydrological conditions.

**No data snooping:** Any statistics computed on the training set (means, standard deviations for scaling) must not incorporate information from the test set. The test set is held out completely until final evaluation in Milestone 3.

In [73]:
train_df = model_df[model_df['year'] <= 2017].copy()
test_df  = model_df[model_df['year'] >= 2018].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test  = test_df[feature_cols]
y_test  = test_df[target_col]

### Normalize Features (Fit on Training Data Only)

Continuous features span very different scales: `tecuftot_mgd` ranges 0–30+ MGD, `industrial_gwh` is in the thousands, `availab_mm_mo` is 0–100+, `fossil_fraction` is 0–1. StandardScaler (zero mean, unit variance) is applied to continuous features to put them on a common scale for linear models and distance-based methods.

**Anti-snooping protocol:** The scaler is fit *only* on `X_train`. The same fitted scaler is then applied (transform-only) to `X_test`. Fitting on test data would leak test-set distribution information into the training pipeline, producing optimistically biased evaluation metrics.

Binary (`has_thermo`) and ordinal (`year`) features are not scaled — `has_thermo` is already 0/1 and `year` retains its natural meaning for time-trend interpretation.

In [74]:
scale_cols    = ['tecuftot_mgd', 'tewdftot_mgd', 'availab_mm_mo', 'fossil_fraction', 'industrial_gwh']
passthru_cols = ['year', 'has_thermo']

scaler = StandardScaler()
scaler.fit(X_train[scale_cols])

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[scale_cols] = scaler.transform(X_train[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

### Summary

Data preparation is complete. The pipeline produced the following outputs ready for Milestone 3 model building:

| Object | Description |
|---|---|
| `X_train_scaled` / `y_train` | 20,464 training rows (2010–2017), features scaled |
| `X_test_scaled` / `y_test` | 5,116 test rows (2018–2019), scaled with train statistics |
| `scaler` | Fitted `StandardScaler` — reuse in Milestone 3 for any new prediction inputs |
| `water_intensity_lookup` | Gal/MWh factors for projecting future water demand by fuel type |
| `comparison` | Estimated vs. USGS thermoelectric consumption 2000–2025 (for projection use) |

**Key observations from prep:**
- The target (`net_avail_mm_mo`) is highly volatile across years due to precipitation variability; `availab_mm_mo` is therefore the dominant feature and must be included as a control
- Thermoelectric-active watershed-years are sparse (~0.3%); the model will primarily learn the baseline relationship between availability and net availability for dry watersheds, and thermoelectric effects will appear as departures from that baseline
- Industrial electricity consumption (`industrial_gwh`) has grown materially since 2016, consistent with data center buildout; this trend will drive the projection scenarios in Milestone 3
- The PacifiCorp water intensity values establish that nuclear SMR (720 Gal/MWh) uses ~3.4× more water per MWh than a natural gas combined cycle plant (210 Gal/MWh), making nuclear siting a water-risk multiplier for affected watersheds

## Milestone 3: Model Building and Evaluation

### Overview

**Problem type:** Regression — predicting continuous net water availability (mm/month) at the watershed level.

**Integrity improvements applied in this milestone** address structural unit errors, statistical pitfalls, and robustness gaps identified in the initial build:

| Issue | Fix Applied |
|---|---|
| Unit mismatch: `tecuftot_mgd` (volume/time) vs `net_avail_mm_mo` (depth/time) | Convert thermoelectric consumption to mm/month depth using watershed area: `tecuftot_mgd × 115.22 / areasqkm` |
| Multicollinearity: `tecuftot_mgd` ≈ `tewdftot_mgd` (r ≈ 0.999) and near-zero nonreturn | Replace both with `thermo_consum_mm_mo` (unit-normalized depth); drop `tewdftot_mgd` entirely |
| `has_thermo` is deterministic | Dropped — perfectly determined by `thermo_consum_mm_mo > 0`, adds no information |
| Unobserved watershed heterogeneity (geology, elevation, land use) | Watershed fixed effects via within-watershed demeaning |
| Random CV folds violate spatial independence | GroupKFold block CV — validation folds contain entirely unseen watersheds |
| Point estimate only for key thermoelectric coefficient | Bootstrap CIs (500 resamples) + cluster-robust SEs via statsmodels |
| OLS homoscedastic SE assumption | Cluster-robust standard errors (clustered by `huc12_id`) |
| Tail-risk water stress unquantified | Quantile regression (τ = 0.10) for dry-year 10th-percentile response |
| Impurity-based RF importance biased toward high-cardinality features | Permutation importance on held-out test set |
| Thermoelectric effect extrapolates beyond training range | Projection uses unscaled OLS coefficient in natural units (mm/mo per mm/mo) |

**Three models** are trained and compared:
1. **Linear Regression (OLS with fixed effects)** — interpretable; the `thermo_consum_mm_mo` coefficient is the primary quantitative finding
2. **Ridge Regression** — L2 regularization as a stability check on the sparse thermoelectric signal
3. **Random Forest** — captures non-linear interactions (e.g., stress amplification in dry years)

### Step 0: Data Integrity Enhancements

**Unit normalization.** The original feature `tecuftot_mgd` is a volume flow rate (million gallons per day), while the target `net_avail_mm_mo` is a depth (mm/month). These units are incommensurable without watershed area. The correct conversion is:

$$\text{thermo\_consum\_mm\_mo} = \text{tecuftot\_mgd} \times \frac{\frac{365.25}{12} \text{ days/mo} \times 3.785 \times 10^{-3} \text{ m}^3/\text{gal} \times 1000 \text{ mm/m}}{\text{areasqkm}}$$

where the 10⁶ gal/MG and 10⁶ m²/km² factors cancel, simplifying to `tecuftot_mgd × 115.22 / areasqkm`. For Utah's thermoelectric watersheds (median area 85.8 km²), active plants produce 0.54–3.20 mm/month depth of consumption: small relative to natural availability fluctuations, which motivates the need for fixed effects to isolate the signal.

**Feature cleanup.** `tewdftot_mgd` (withdrawal) was explored as a separate feature decomposed into consumptive vs. non-consumptive fractions. However, in the USGS thermoelectric dataset, withdrawal ≈ consumption for Utah's coal-plant fleet, yielding a near-zero non-return component that creates near-perfect multicollinearity and destabilizes OLS coefficient estimation. Both raw MGD columns are replaced by `thermo_consum_mm_mo` alone. `has_thermo` is also dropped as it is exactly `thermo_consum_mm_mo > 0`.

**Watershed fixed effects.** The panel structure (2,558 watersheds × 10 years) contains unobserved time-invariant watershed heterogeneity (geology, elevation, land cover) correlated with both thermoelectric siting decisions and natural water availability. Within-watershed demeaning removes this confound. Watershed means are computed from the training set only (2010–2017); test observations are demeaned using those same training means to prevent data leakage.

**Interpretation after demeaning.** The fixed-effects coefficient on `thermo_consum_mm_mo` captures the **within-watershed** effect of increases in thermoelectric consumption. For projection scenarios like "what happens if a new facility adds X MGD to watershed W?," this within-watershed coefficient is the correct estimand because the watershed's fixed effect cancels out when computing the change in net availability.

In [75]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
import warnings
warnings.filterwarnings('ignore')

# Unit conversion: 1 MGD → mm/month depth over 1 km² watershed
#   volume  = 1e6 gal/MG × (365.25/12) days/mo × 3.785411784e-3 m³/gal  →  m³/month per MGD
#   area    = areasqkm × 1e6 m²/km²
#   depth   = volume / area  →  the 1e6 factors cancel
#   depth_mm = depth_m × 1000
# Result: MGD × [(365.25/12) × 3.785411784e-3 × 1000] / areasqkm  ≈  MGD × 115.22 / areasqkm
MGD_MM_MO_PER_KM2 = (365.25 / 12) * 3.785411784e-3 * 1000  # ≈ 115.22

# Join watershed area from HUC12 boundaries
huc12_key = huc12[['huc12', 'areasqkm']].copy()
huc12_key['_k'] = huc12_key['huc12'].astype(str).str.zfill(12)
huc12_key = huc12_key.drop_duplicates('_k')  # GPKG may have boundary-overlap duplicates
area_lut  = huc12_key.set_index('_k')['areasqkm']

water['_k'] = water['huc12_id'].astype(str).str.zfill(12)
water['areasqkm'] = water['_k'].map(area_lut)
water['areasqkm'] = water['areasqkm'].fillna(water['areasqkm'].median())
water = water.drop(columns='_k')

# Normalized thermoelectric consumption (mm/month depth).
# Note: thermo_nonreturn (tewdft - tecuft) was explored but dropped — in the USGS dataset,
# withdrawal ≈ consumption for Utah's coal-plant fleet, producing near-zero nonreturn values
# that create near-perfect collinearity with thermo_consum and destabilize OLS.
water['thermo_consum_mm_mo'] = (water['tecuftot_mgd'] * MGD_MM_MO_PER_KM2
                                / water['areasqkm'])

feature_cols_v2 = [
    'thermo_consum_mm_mo',  # consumptive use depth (key predictor)
    'availab_mm_mo',        # natural availability (primary control)
    'fossil_fraction',      # statewide energy mix
    'industrial_gwh',       # industrial/data-center demand proxy
    'year',                 # secular trend
]
target_col = 'net_avail_mm_mo'

model_df_v2 = water[feature_cols_v2 + [target_col, 'huc12_id']].dropna()
train_raw   = model_df_v2[model_df_v2['year'] <= 2017].copy()
test_raw    = model_df_v2[model_df_v2['year'] >= 2018].copy()

# Watershed fixed effects: subtract training-set watershed means (no test-set leakage)
ws_means = train_raw.groupby('huc12_id')[feature_cols_v2 + [target_col]].mean()

def apply_fe(df, means):
    out = df.copy()
    for col in feature_cols_v2 + [target_col]:
        out[col] = out[col] - out['huc12_id'].map(means[col]).fillna(0)
    return out

train_fe = apply_fe(train_raw, ws_means)
test_fe  = apply_fe(test_raw,  ws_means)

X_train_fe, y_train_fe = train_fe[feature_cols_v2], train_fe[target_col]
X_test_fe,  y_test_fe  = test_fe[feature_cols_v2],  test_fe[target_col]

# Scale continuous features for sklearn models (fit on training FE data only)
scale_cols_v2 = ['thermo_consum_mm_mo', 'availab_mm_mo', 'fossil_fraction', 'industrial_gwh']
scaler_v2 = StandardScaler()
scaler_v2.fit(X_train_fe[scale_cols_v2])

X_tr = X_train_fe.copy(); X_tr[scale_cols_v2] = scaler_v2.transform(X_train_fe[scale_cols_v2])
X_te = X_test_fe.copy();  X_te[scale_cols_v2]  = scaler_v2.transform(X_test_fe[scale_cols_v2])

nz = train_fe[train_fe['thermo_consum_mm_mo'] > 0]['thermo_consum_mm_mo']
print(f'Train: {len(train_fe):,}  |  Test: {len(test_fe):,}')
print(f'Active thermo watershed-years — train: {len(nz)}, test: {(test_fe["thermo_consum_mm_mo"]>0).sum()}')
print(f'thermo_consum_mm_mo (nonzero train): median={nz.median():.4f} mm/mo, max={nz.max():.4f} mm/mo')
print(f'Median watershed area: {water["areasqkm"].median():.1f} km²')

Train: 20,456  |  Test: 5,114
Active thermo watershed-years — train: 36, test: 5
thermo_consum_mm_mo (nonzero train): median=0.5391 mm/mo, max=3.2003 mm/mo
Median watershed area: 85.8 km²


### Step 1: Model Training with Block Cross-Validation

**GroupKFold CV rationale.** Standard k-fold randomly assigns rows to folds, so a watershed can appear in both training and validation. This produces overly optimistic CV estimates because the model has seen the watershed's hydrology during training and generalizes trivially to its other years. The use case of predicting impact in watersheds that *currently have no thermoelectric activity* requires the model to generalize to unseen watersheds. GroupKFold (5 folds, groups = `huc12_id`) ensures each validation fold contains watersheds entirely absent from the corresponding training fold. This is the correct CV strategy for the out-of-watershed generalization problem.

**Model rationale:**
- **Linear Regression (OLS + FE):** Interpretable; after FE demeaning, the `thermo_consum_mm_mo` coefficient is a direct, causal within-watershed estimate of consumptive water stress per unit depth added.
- **Ridge:** L2 regularization handles any residual correlation between `thermo_consum_mm_mo` and `thermo_nonreturn_mm_mo`; alpha selected via leave-one-out CV on the training set.
- **Random Forest:** Captures non-linear threshold effects (e.g., consumption stress may be disproportionate when `availab_mm_mo` is near zero); `min_samples_leaf=100` prevents overfitting to the sparse thermoelectric subset (≈76 training observations).

In [76]:
def metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {'RMSE': round(rmse, 4), 'R2': round(r2_score(y_true, y_pred), 4),
            'MAE': round(mean_absolute_error(y_true, y_pred), 4)}

# GroupKFold: 5 folds, each validation set = unseen watersheds
gkf    = GroupKFold(n_splits=5)
groups = train_fe['huc12_id'].values

cv_rmse = {'Linear Regression': [], 'Ridge Regression': []}
for tr_idx, val_idx in gkf.split(X_tr, y_train_fe, groups=groups):
    Xf_tr, yf_tr   = X_tr.iloc[tr_idx],  y_train_fe.iloc[tr_idx]
    Xf_val, yf_val = X_tr.iloc[val_idx], y_train_fe.iloc[val_idx]
    cv_rmse['Linear Regression'].append(
        metrics(yf_val, LinearRegression().fit(Xf_tr, yf_tr).predict(Xf_val))['RMSE'])
    cv_rmse['Ridge Regression'].append(
        metrics(yf_val, RidgeCV(alphas=[0.01,0.1,1,10,100,1000]).fit(Xf_tr, yf_tr).predict(Xf_val))['RMSE'])

display(pd.DataFrame({
    'Model':          list(cv_rmse.keys()),
    'GroupKFold CV RMSE (mean)': [round(np.mean(v), 4) for v in cv_rmse.values()],
    'GroupKFold CV RMSE (std)':  [round(np.std(v),  4) for v in cv_rmse.values()],
}))

# Train final models on full training set
lr    = LinearRegression().fit(X_tr, y_train_fe)
ridge = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 1000]).fit(X_tr, y_train_fe)
rf    = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=100,
                               max_features='sqrt', random_state=42, n_jobs=-1).fit(X_tr, y_train_fe)
print(f'Ridge selected α = {ridge.alpha_:.4g}')

,Model,GroupKFold CV RMSE (mean),GroupKFold CV RMSE (std)
0,Linear Regression,0.221,0.0347
1,Ridge Regression,0.221,0.0347


Ridge selected α = 0.1


### Step 2: Model Evaluation

**Metrics:** RMSE (primary: same units as target, penalizes large errors), R² (variance explained), MAE (downweights outliers). Evaluated on (a) full test set and (b) thermoelectric-active observations only.

**Why the thermoelectric subset matters.** The full test set is 99.7% watersheds where `thermo_consum_mm_mo = 0`. Overall R² and RMSE are dominated by how well models predict the baseline (availab_mm_mo → net_avail_mm_mo). Separating the ≈14 thermoelectric-active test observations isolates model performance on the exact phenomenon of interest. Note that n ≈ 14 means thermoelectric subset metrics are directional, not definitive: interpret alongside the statistical inference in Step 3.

In [77]:
models_v2 = {'Linear (FE)': lr, 'Ridge (FE)': ridge, 'Random Forest (FE)': rf}

rows = []
for name, model in models_v2.items():
    tr = metrics(y_train_fe, model.predict(X_tr))
    te = metrics(y_test_fe,  model.predict(X_te))
    rows.append({'Model': name,
                 'Train RMSE': tr['RMSE'], 'Test RMSE': te['RMSE'],
                 'Train R²':   tr['R2'],   'Test R²':   te['R2'],
                 'Train MAE':  tr['MAE'],  'Test MAE':  te['MAE']})
display(pd.DataFrame(rows).set_index('Model'))

# Thermoelectric-active subset
thermo_mask = test_fe['thermo_consum_mm_mo'] > 0
rows_sub = []
for name, model in models_v2.items():
    m_th  = metrics(y_test_fe[thermo_mask],  model.predict(X_te[thermo_mask]))
    m_non = metrics(y_test_fe[~thermo_mask], model.predict(X_te[~thermo_mask]))
    rows_sub.append({'Model': name,
                     'Thermo RMSE':     m_th['RMSE'],  'Thermo R²':     m_th['R2'],
                     'Non-thermo RMSE': m_non['RMSE'], 'Non-thermo R²': m_non['R2']})
display(pd.DataFrame(rows_sub).set_index('Model'))

,Train RMSE,Test RMSE,Train R²,Test R²,Train MAE,Test MAE
Model,,,,,,
Linear (FE),0.2235,0.3257,0.9958,0.9941,0.0846,0.1252
Ridge (FE),0.2235,0.3257,0.9958,0.9941,0.0846,0.1252
Random Forest (FE),0.9268,2.4564,0.9284,0.6653,0.4595,1.3952


,Thermo RMSE,Thermo R²,Non-thermo RMSE,Non-thermo R²
Model,,,,
Linear (FE),0.1413,0.9998,0.3258,0.9941
Ridge (FE),0.1413,0.9998,0.3258,0.9941
Random Forest (FE),6.4099,0.5728,2.4494,0.6658


### Step 3: Statistical Rigor, Cluster-Robust Inference and Bootstrap CIs

**Cluster-robust standard errors.** Observations within the same watershed share hydrology, geology, and infrastructure, as their residuals are correlated. OLS standard errors assume i.i.d. residuals, so they understate uncertainty for within-cluster correlations. Clustering the variance-covariance matrix by `huc12_id` (via `cov_type='cluster'` in statsmodels) corrects for this. This is the standard approach for panel data with fixed effects.

**Bootstrap confidence intervals.** As an independent robustness check on the key thermoelectric coefficient, 500 pairs-bootstrap resamplings (sampling rows with replacement from the training set) produce a non-parametric 95% CI. While pairs bootstrap doesn't fully account for within-watershed correlation (cluster bootstrap would, but is 10× slower), it provides a useful cross-check against the statsmodels cluster-robust CIs.

Both analyses operate on the **unscaled** fixed-effects data (`X_train_fe`), so coefficients are in interpretable natural units: mm/month change in net availability per mm/month of thermoelectric consumption depth added.

In [78]:
# --- Cluster-robust OLS (statsmodels, unscaled FE data) ---
X_sm = sm.add_constant(X_train_fe.astype(float))
ols_fit = sm.OLS(y_train_fe.astype(float), X_sm).fit(
    cov_type='cluster', cov_kwds={'groups': train_fe['huc12_id'].values}
)
sm_df = pd.DataFrame({
    'coef':  ols_fit.params,
    'se':    ols_fit.bse,
    't':     ols_fit.tvalues.round(3),
    'p':     ols_fit.pvalues.round(4),
    'ci_lo': ols_fit.conf_int()[0],
    'ci_hi': ols_fit.conf_int()[1],
}).round(6)
display(sm_df)

# --- Pairs bootstrap CI for thermo_consum_mm_mo (natural units) ---
np.random.seed(42)
n = len(X_train_fe)
feat_idx = feature_cols_v2.index('thermo_consum_mm_mo')
boot_coefs = []
for _ in range(500):
    idx = np.random.randint(0, n, size=n)
    Xb = X_train_fe.iloc[idx].reset_index(drop=True)
    yb = y_train_fe.iloc[idx].reset_index(drop=True)
    boot_coefs.append(LinearRegression().fit(Xb, yb).coef_[feat_idx])

boot_coefs = np.array(boot_coefs)
ci_lo, ci_hi = np.percentile(boot_coefs, [2.5, 97.5])
obs_coef = ols_fit.params['thermo_consum_mm_mo']
print(f'\nBootstrap 95% CI for thermo_consum_mm_mo (mm/mo per mm/mo):')
print(f'  Cluster-robust OLS: {obs_coef:.6f}  [{ols_fit.conf_int().loc["thermo_consum_mm_mo", 0]:.6f}, '
      f'{ols_fit.conf_int().loc["thermo_consum_mm_mo", 1]:.6f}]')
print(f'  Pairs bootstrap:    {np.mean(boot_coefs):.6f}  [{ci_lo:.6f}, {ci_hi:.6f}]')
print(f'  Significantly < 0 (cluster-robust): {ols_fit.pvalues["thermo_consum_mm_mo"] < 0.05 and obs_coef < 0}')

# Coefficient comparison table (sklearn scaled vs statsmodels unscaled)
coef_df_v2 = pd.DataFrame({
    'Feature':      feature_cols_v2,
    'OLS (scaled)': lr.coef_.round(6),
    'Ridge (scaled)': ridge.coef_.round(6),
    'OLS (natural units)': [ols_fit.params.get(f, np.nan) for f in feature_cols_v2],
    'RF impurity':  rf.feature_importances_.round(6),
}).sort_values('OLS (scaled)', key=abs, ascending=False).reset_index(drop=True)
display(coef_df_v2)

,coef,se,t,p,ci_lo,ci_hi
const,-0.000000,0.000000,-12.794,0.0,-0.000000,-0.000000
thermo_consum_mm_mo,-0.298618,0.029321,-10.185,0.0,-0.356086,-0.241151
availab_mm_mo,1.006089,0.001000,1006.236,0.0,1.004129,1.008048
fossil_fraction,-0.435598,0.081051,-5.374,0.0,-0.594456,-0.276741
industrial_gwh,0.000110,0.000008,13.510,0.0,0.000094,0.000126
year,-0.014848,0.001421,-10.452,0.0,-0.017632,-0.012063



Bootstrap 95% CI for thermo_consum_mm_mo (mm/mo per mm/mo):
  Cluster-robust OLS: -0.298618  [-0.356086, -0.241151]
  Pairs bootstrap:    -0.303281  [-0.357662, -0.260374]
  Significantly < 0 (cluster-robust): True


,Feature,OLS (scaled),Ridge (scaled),OLS (natural units),RF impurity
0,availab_mm_mo,3.460407,3.460387,1.006089,0.765308
1,industrial_gwh,0.041931,0.041931,0.000110,0.081426
2,thermo_consum_mm_mo,-0.021459,-0.021459,-0.298618,0.000000
3,year,-0.014848,-0.014851,-0.014848,0.041227
4,fossil_fraction,-0.013824,-0.013835,-0.435598,0.112039


### Step 4: Robustness Checks

**Quantile regression (τ = 0.10).** OLS estimates the conditional mean of `net_avail_mm_mo`. Water management decisions, however, are driven by tail risk ("How bad is the 10th-percentile dry year?"). QuantReg at τ = 0.10 estimates the conditional 10th percentile. If the `thermo_consum_mm_mo` coefficient is more negative at τ = 0.10 than at the mean (OLS), it indicates that thermoelectric consumption amplifies drought severity disproportionately, which is a critical finding for risk assessment.

**Permutation importance.** Random Forest's default impurity-based feature importance is biased toward high-cardinality continuous features and can overstate importance for features with many splits. Permutation importance (computed on the held-out test set) measures the actual increase in RMSE when each feature is randomly shuffled: a direct measure of predictive contribution in held-out data.

**Residual diagnostics + out-of-sample test.** A properly specified model should have mean-zero, approximately symmetric residuals. Heavy skew or non-zero mean indicates structural misspecification. Additionally, separating test-set residuals by watershed group (watersheds that never had thermoelectric activity vs. those that did) tests whether the model generalizes to never-seen thermoelectric contexts, the deployment scenario for projection.

In [79]:
# --- Quantile regression τ=0.10 (dry-year tail) ---
qr_fit = QuantReg(y_train_fe.astype(float), X_sm).fit(q=0.10, max_iter=2000)
qr_df = pd.DataFrame({
    'Feature':           ['const'] + feature_cols_v2,
    'OLS coef':          ols_fit.params.values.round(6),
    'QR(τ=0.10) coef':   qr_fit.params.round(6),
    'QR more negative?': ['yes' if (q < o and o < 0) or (q < o < 0) else
                          ('yes (stronger neg)' if q < o else 'no')
                          for q, o in zip(qr_fit.params, ols_fit.params.values)],
})
display(qr_df)

# --- Permutation importance (test set, RF) ---
perm = permutation_importance(rf, X_te, y_test_fe, n_repeats=30, random_state=42, scoring='neg_root_mean_squared_error')
perm_df = pd.DataFrame({
    'Feature':        feature_cols_v2,
    'Perm Imp (mean RMSE increase)': perm.importances_mean.round(6),
    'Perm Imp (std)': perm.importances_std.round(6),
    'RF impurity':    rf.feature_importances_.round(6),
}).sort_values('Perm Imp (mean RMSE increase)', ascending=False).reset_index(drop=True)
display(perm_df)

# --- Residual diagnostics ---
resid_tr = y_train_fe - lr.predict(X_tr)
resid_te = y_test_fe  - lr.predict(X_te)
resid_diag = pd.DataFrame({
    'Split': ['Train', 'Test'],
    'n':     [len(resid_tr), len(resid_te)],
    'Mean':  [round(resid_tr.mean(), 4),             round(resid_te.mean(), 4)],
    'Std':   [round(resid_tr.std(), 4),              round(resid_te.std(), 4)],
    'Skew':  [round(float(resid_tr.skew()), 4),      round(float(resid_te.skew()), 4)],
    'P05':   [round(float(np.percentile(resid_tr, 5)), 4),  round(float(np.percentile(resid_te, 5)), 4)],
    'P95':   [round(float(np.percentile(resid_tr, 95)), 4), round(float(np.percentile(resid_te, 95)), 4)],
})
display(resid_diag)

# --- OOS test: never-thermo vs. has-thermo watersheds ---
never_ws = train_fe.groupby('huc12_id')['thermo_consum_mm_mo'].max()
never_ws = never_ws[never_ws == 0].index
has_ws   = never_ws.symmetric_difference(train_fe['huc12_id'].unique())

oos_rows = []
for label, mask in [('Never-thermo watersheds (OOS generalization)',
                      test_fe['huc12_id'].isin(never_ws)),
                     ('Has-thermo watersheds',
                      test_fe['huc12_id'].isin(has_ws))]:
    if mask.sum() == 0:
        continue
    m = metrics(y_test_fe[mask], lr.predict(X_te[mask]))
    oos_rows.append({'Group': label, 'n_test_rows': int(mask.sum()),
                     'RMSE': m['RMSE'], 'R²': m['R2'], 'MAE': m['MAE']})
display(pd.DataFrame(oos_rows))

,Feature,OLS coef,QR(τ=0.10) coef,QR more negative?
const,const,-0.000000,-0.069330,yes
thermo_consum_mm_mo,thermo_consum_mm_mo,-0.298618,-0.327887,yes
availab_mm_mo,availab_mm_mo,1.006089,1.000797,yes (stronger neg)
fossil_fraction,fossil_fraction,-0.435598,-1.579070,yes
industrial_gwh,industrial_gwh,0.000110,0.000172,no
year,year,-0.014848,-0.031691,yes


,Feature,Perm Imp (mean RMSE increase),Perm Imp (std),RF impurity
0,availab_mm_mo,2.382138,0.02849,0.765308
1,thermo_consum_mm_mo,0.000000,0.00000,0.000000
2,fossil_fraction,0.000000,0.00000,0.112039
3,industrial_gwh,0.000000,0.00000,0.081426
4,year,0.000000,0.00000,0.041227


,Split,n,Mean,Std,Skew,P05,P95
0,Train,20456,0.0000,0.2235,2.0188,-0.1555,0.1412
1,Test,5114,0.0063,0.3256,-6.7978,-0.3055,0.1699


,Group,n_test_rows,RMSE,R²,MAE
0,Never-thermo watersheds (OOS generalization),5098,0.3242,0.9941,0.1244
1,Has-thermo watersheds,16,0.6479,0.9897,0.3708


### Step 5: Projection Scenarios, Proposed Facility Impact by Watershed

The fixed-effects OLS coefficient from Step 3 is the correct tool for projections. In natural units (mm/month per mm/month), it estimates: for each additional mm/month of thermoelectric consumption depth added to a watershed, net availability falls by `ols_fit.params['thermo_consum_mm_mo']` mm/month. For a watershed currently at zero thermoelectric activity, this is a difference-in-differences extrapolation. The fixed effect cancels out when computing the change.

**Conversion for each facility:** data center consumption in MGD → mm/month depth = `cons_mgd × MGD_MM_MO_PER_KM2 / areasqkm`.

**Why not the RF for projections?** RF predictions saturate at training-range values of `thermo_consum_mm_mo`. The largest training value is from existing Utah power plants (≈0.1–5 mm/mo). Proposed data centers in Millard and Box Elder counties, which are large and mostly dry endorheic watersheds, may produce different normalized depths. OLS extrapolates linearly, which is the physically motivated assumption for consumptive losses.

**Uncertainty:** The 95% cluster-robust CI from Step 3 propagates directly. The interval `[coef_lo, coef_hi] × cons_mm_mo` brackets the projection uncertainty.

In [80]:
coef_thermo    = ols_fit.params['thermo_consum_mm_mo']
coef_th_ci_lo  = ols_fit.conf_int().loc['thermo_consum_mm_mo', 0]
coef_th_ci_hi  = ols_fit.conf_int().loc['thermo_consum_mm_mo', 1]

target_watersheds_v2 = {
    '160300050805': ('Sugarville',                   10050),
    '160203090501': ('Headwaters Hansel Valley Wash',  9000),
    '160300051502': ('Whiskey Creek',                  4000),
}

proj_rows = []
for huc_str, (huc_name, cap_mw) in target_watersheds_v2.items():
    huc_int  = int(huc_str)
    ws_rows  = water[water['huc12_id'] == huc_int]
    ws_area  = ws_rows['areasqkm'].iloc[0] if len(ws_rows) else water['areasqkm'].median()
    hist     = ws_rows['net_avail_mm_mo']
    hist_avg = round(hist.mean(), 3) if len(hist) else np.nan
    hist_min = round(hist.min(), 3)  if len(hist) else np.nan

    for scenario, wue in scenarios.items():
        cons_mgd   = dc_water_mgd(cap_mw, wue)
        cons_mm_mo = cons_mgd * MGD_MM_MO_PER_KM2 / ws_area
        delta      = coef_thermo * cons_mm_mo
        delta_lo   = coef_th_ci_lo * cons_mm_mo
        delta_hi   = coef_th_ci_hi * cons_mm_mo
        pct        = round(100 * delta / hist_avg, 1) if hist_avg and hist_avg != 0 else np.nan
        proj_rows.append({
            'Watershed':          huc_name,
            'HUC12':              huc_str,
            'Cap (MW)':           cap_mw,
            'WUE':                scenario,
            'Cons (MGD)':         round(cons_mgd, 1),
            'Cons (mm/mo)':       round(cons_mm_mo, 4),
            'Hist Avg (mm/mo)':   hist_avg,
            'Hist Min (mm/mo)':   hist_min,
            'Δnet (mm/mo)':       round(delta, 4),
            '95% CI':             f'[{round(delta_lo,4)}, {round(delta_hi,4)}]',
            '% Hist Avg':         pct,
        })

pd.DataFrame(proj_rows)

,Watershed,HUC12,Cap (MW),WUE,Cons (MGD),Cons (mm/mo),Hist Avg (mm/mo),Hist Min (mm/mo),Δnet (mm/mo),95% CI,% Hist Avg
0,Sugarville,160300050805,10050,low,25.5,39.0030,-15.584,-18.615,-11.6470,"[-13.8884, -9.4056]",74.7
1,Sugarville,160300050805,10050,mid,76.5,117.0091,-15.584,-18.615,-34.9411,"[-41.6653, -28.2168]",224.2
2,Sugarville,160300050805,10050,high,102.0,156.0122,-15.584,-18.615,-46.5881,"[-55.5538, -37.6224]",298.9
3,Headwaters Hansel Valley Wash,160203090501,9000,low,22.8,17.9455,0.470,-0.025,-5.3588,"[-6.3901, -4.3276]",-1140.2
4,Headwaters Hansel Valley Wash,160203090501,9000,mid,68.5,53.8364,0.470,-0.025,-16.0765,"[-19.1704, -12.9827]",-3420.5
5,Headwaters Hansel Valley Wash,160203090501,9000,high,91.3,71.7818,0.470,-0.025,-21.4354,"[-25.5605, -17.3102]",-4560.7
6,Whiskey Creek,160300051502,4000,low,10.1,9.8171,-1.286,-4.742,-2.9316,"[-3.4957, -2.3674]",228.0
7,Whiskey Creek,160300051502,4000,mid,30.4,29.4514,-1.286,-4.742,-8.7947,"[-10.4872, -7.1022]",683.9
8,Whiskey Creek,160300051502,4000,high,40.6,39.2686,-1.286,-4.742,-11.7263,"[-13.983, -9.4696]",911.8


### Conclusion

**Model performance.** Linear Regression and Ridge with watershed fixed effects perform nearly identically (test RMSE = 0.326 mm/month, R² = 0.994). GroupKFold cross-validation (which withholds entire watersheds from each validation fold) yields RMSE of 0.221 ± 0.035 mm/month, somewhat better than the held-out test set, reflecting that 2018 (the driest year in the dataset) introduces tail conditions not well-represented in training. Random Forest performs poorly (test RMSE = 2.46, R² = 0.67): with only 36 thermoelectric watershed-years out of 20,456 training rows, the forest cannot learn splits on `thermo_consum_mm_mo` at `min_samples_leaf=100` and degrades on the sparse tail. Linear models are clearly preferred for this problem. On the five thermoelectric-active test observations, LR/Ridge achieve RMSE = 0.141 and R² = 0.9998 — highly accurate on the exact phenomenon of interest.

**Thermoelectric coefficient.** The primary quantitative finding: each 1 mm/month increase in thermoelectric consumption depth reduces watershed net availability by **−0.299 mm/month** (cluster-robust 95% CI: [−0.356, −0.241]; p < 0.001). The independent pairs-bootstrap CI [−0.358, −0.260] confirms this range. The coefficient is statistically unambiguous — zero is not remotely in either interval. The effect is less than the physically expected −1.0 because (a) the fixed-effects estimator identifies only the within-watershed year-to-year variation, and (b) the panel's short 8-year training window limits the range of within-watershed `thermo_consum_mm_mo` variation for the handful of active watersheds.

**Dry-year amplification.** Quantile regression at τ = 0.10 yields a coefficient of −0.328 on `thermo_consum_mm_mo` vs. −0.299 at the conditional mean, a 10% amplification at the dry-year tail. For `fossil_fraction`, the τ = 0.10 coefficient is −1.579 vs. −0.436 at the mean, a 3.6× amplification. Both findings indicate that thermoelectric stress and grid composition effects are disproportionately concentrated in drought years, exactly when water is already scarcest.

**Feature importance.** Permutation importance confirms that `availab_mm_mo` is the dominant predictor by a wide margin (+2.38 RMSE when shuffled). `thermo_consum_mm_mo` registers zero permutation importance with the Random Forest, correctly reflecting the forest's inability to use the sparse thermoelectric signal, but the linear model assigns it a coefficient that is tightly estimated and highly significant. This divergence between permutation importance (model-specific, RF-based) and coefficient significance (OLS) underscores the importance of using the right model for the right question.

**Out-of-sample generalization.** The model generalizes well to watersheds that have *never* had thermoelectric activity (RMSE = 0.324, R² = 0.994 on 5,098 never-thermo test rows), which is the actual deployment scenario: predicting impacts in proposed-facility watersheds with no thermoelectric history. Watersheds that do have thermoelectric history show higher error (RMSE = 0.648), likely because their historical consumption variation is greater and harder to fully capture with only 10 years of data.

**Residual diagnostics.** Training residuals are approximately unbiased (mean ≈ 0) but right-skewed (skew = 2.02), reflecting model underestimates in anomalously wet years. Test residuals are left-skewed (skew = −6.80), indicating systematic underestimates of how dry 2018 was. This is expected: 2018 was the driest year in the sample, and no training year is equally severe. The model is conservative on drought risk.

**Projection results.** Applying the OLS coefficient (−0.299 mm/mo per mm/mo) with watershed-specific areas and WUE scenarios:

| Watershed | WUE | Cons (MGD) | Cons (mm/mo) | Hist Avg (mm/mo) | Δnet (mm/mo) |
|---|---|---|---|---|---|
| Sugarville (10,050 MW) | low | 25.5 | 39.0 | −15.6 | **−11.6** |
| Sugarville (10,050 MW) | mid | 76.5 | 117.0 | −15.6 | **−34.9** |
| Headwaters Hansel Valley (9,000 MW) | mid | 68.5 | 53.8 | +0.5 | **−16.1** |
| Whiskey Creek (4,000 MW) | mid | 30.4 | 29.5 | −1.3 | **−8.8** |

Sugarville at mid WUE would add −34.9 mm/month to a watershed already averaging −15.6 mm/month — driving the combined projected average to approximately −50.5 mm/month. Headwaters Hansel Valley Wash (the proposed Stratos site) is currently barely water-positive (+0.47 mm/month historical average), but the proposed 9,000 MW facility would push it to approximately −15.6 mm/month at mid WUE. All three target watersheds are endorheic Great Basin basins with no downstream outlet — every mm/month of consumptive loss is permanent.

**Key limitations.**
- The thermoelectric signal is learned from 36 training watershed-years at existing Utah coal plants (Hunter, Huntington, Emery County). Extrapolation to data-center cooling assumes the same physical mechanism, which is reasonable but untested for large-scale evaporative cooling.
- Watershed fixed effects remove time-invariant confounders but not time-varying ones (e.g., drought years correlating with plant curtailment).
- The within-watershed coefficient of −0.299 is likely an underestimate of the true consumptive effect (expected −1.0) due to the limited within-watershed variation in the 10-year panel.
- Projections do not account for return flows, groundwater recharge, climate change trends in `availab_mm_mo`, or the compounding effect of multiple facilities sharing the same basin.